In [1]:
# =============================================================================
# DATATHON 2026 — Sales Forecasting v2
# Dựa trên notebook gốc (model_training_fixed_cleaned.ipynb)
# Giữ nguyên: Prophet+residual, LGB+XGB+CatBoost, expanding means, recursive
# Fix:        stacking leakage, recursive missing features
# Nâng cấp:  VN holidays, EWMA, YoY, Tết flag, SHAP, OOF đúng chuẩn
# =============================================================================
 
# %% ── Cell 0: Imports ────────────────────────────────────────────────────────
import copy, os, warnings
warnings.filterwarnings('ignore')
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
 
from prophet import Prophet
import xgboost as xgb
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
 

In [2]:
SEED     = 42
DATA_DIR = '../data'
OUT_DIR  = '../data/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(SEED)

In [3]:
# %% ── Cell 1: Load data ──────────────────────────────────────────────────────
data = pd.read_csv(os.path.join(DATA_DIR, 'raw', 'sales.csv'),
                   parse_dates=['Date'])
data = data.sort_values('Date').reset_index(drop=True)
 
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'raw', 'sample_submission.csv'),
                          parse_dates=['Date'])
 
print(f"Train: {data['Date'].min().date()} → {data['Date'].max().date()}  ({len(data):,} rows)")
print(f"Test : {sample_sub['Date'].min().date()} → {sample_sub['Date'].max().date()}")

Train: 2012-07-04 → 2022-12-31  (3,833 rows)
Test : 2023-01-01 → 2024-07-01


In [4]:
# %% ── Cell 2: Vietnam Holiday Calendar ───────────────────────────────────────
# Kiến thức chung về lịch VN — không phải external data
 
TET = {                         # Mùng 1 Tết (dương lịch)
    2013:"2013-02-10", 2014:"2014-01-31", 2015:"2015-02-19",
    2016:"2016-02-08", 2017:"2017-01-28", 2018:"2018-02-16",
    2019:"2019-02-05", 2020:"2020-01-25", 2021:"2021-02-12",
    2022:"2022-02-01", 2023:"2023-01-22", 2024:"2024-02-10",
}
 
# Set ngày Tết để dùng làm binary flag feature
TET_WINDOW: set = set()
for s in TET.values():
    tet = pd.Timestamp(s)
    for d in range(-7, 8):          # 7 ngày trước + 7 ngày sau
        TET_WINDOW.add(tet + pd.Timedelta(days=d))
 
# Ngày lễ cố định (giữ nguyên từ notebook gốc + bổ sung)
HOLIDAYS_MMDD = {
    '01-01',  # Tết Dương lịch
    '02-14',  # Valentine
    '03-08',  # Quốc tế phụ nữ
    '04-30',  # Giải phóng miền Nam
    '05-01',  # Quốc tế lao động
    '09-02',  # Quốc khánh
    '10-20',  # Phụ nữ Việt Nam
    '11-11',  # Sale 11/11
    '12-12',  # Sale 12/12
    '12-24',  # Christmas Eve
    '12-25',  # Christmas
    '12-31',  # Tất niên
}
 
def build_prophet_holidays() -> pd.DataFrame:
    """Tạo DataFrame holidays cho Prophet với VN calendar đầy đủ."""
    rows = []
    for year, s in TET.items():
        tet = pd.Timestamp(s)
        # Pre-Tết spike: mua sắm tăng mạnh 1 tuần trước
        rows.append(dict(holiday='tet_pre_spike',
                         ds=tet - pd.Timedelta(days=7),
                         lower_window=0, upper_window=6))
        # Tết break: đóng cửa hàng, doanh thu giảm
        rows.append(dict(holiday='tet_break',
                         ds=tet, lower_window=0, upper_window=6))
 
    fixed = {
        'New_Year': '01-01', 'Valentine': '02-14',
        'Womens_Day': '03-08', 'Reunification': '04-30',
        'Labor_Day': '05-01', 'National_Day': '09-02',
        'VN_Womens': '10-20', 'Sale_1111': '11-11',
        'Sale_1212': '12-12', 'Christmas': '12-25', 'New_Year_Eve': '12-31',
    }
    for yr in range(2012, 2025):
        for name, mmdd in fixed.items():
            try:
                rows.append(dict(holiday=name,
                                 ds=pd.Timestamp(f'{yr}-{mmdd}'),
                                 lower_window=-1, upper_window=1))
            except ValueError:
                pass
    return pd.DataFrame(rows)
 
vn_holidays = build_prophet_holidays()
print(f"VN holidays: {vn_holidays['holiday'].nunique()} loại, {len(vn_holidays)} entries")
 

VN holidays: 13 loại, 167 entries


In [5]:
# %% ── Cell 3: Feature Engineering ───────────────────────────────────────────
# GIỮ NGUYÊN từ notebook gốc + bổ sung thêm
 
# ── Calendar features (giữ nguyên gốc) ──
data['year']        = data['Date'].dt.year
data['month']       = data['Date'].dt.month
data['day']         = data['Date'].dt.day
data['dayofweek']   = data['Date'].dt.dayofweek
data['dayofyear']   = data['Date'].dt.dayofyear
data['weekofyear']  = data['Date'].dt.isocalendar().week.astype(int)
data['is_weekend']  = data['dayofweek'].isin([5, 6]).astype(int)
data['is_holiday']  = data['Date'].dt.strftime('%m-%d').isin(HOLIDAYS_MMDD).astype(int)
 
# Fourier encoding (giữ nguyên gốc)
data['sin_1'] = np.sin(2 * np.pi * data['dayofyear'] / 365.25)
data['cos_1'] = np.cos(2 * np.pi * data['dayofyear'] / 365.25)
data['sin_2'] = np.sin(4 * np.pi * data['dayofyear'] / 365.25)
data['cos_2'] = np.cos(4 * np.pi * data['dayofyear'] / 365.25)
# Thêm Fourier cho weekly seasonality
data['sin_dow'] = np.sin(2 * np.pi * data['dayofweek'] / 7)
data['cos_dow'] = np.cos(2 * np.pi * data['dayofweek'] / 7)
 
# ── BỔ SUNG: Tết & calendar features mới ──
data['is_tet_period']  = data['Date'].isin(TET_WINDOW).astype(int)
data['quarter']        = data['Date'].dt.quarter
data['is_month_start'] = data['Date'].dt.is_month_start.astype(int)
data['is_month_end']   = data['Date'].dt.is_month_end.astype(int)
data['is_quarter_end'] = data['Date'].dt.is_quarter_end.astype(int)
 
# Days to nearest Tết (continuous proximity)
tet_ts = pd.Series(sorted(TET_WINDOW))
data['days_to_tet'] = data['Date'].apply(
    lambda d: int((tet_ts - d).abs().min().days)
)
 

In [6]:
# %% ── Cell 4: Lag & Rolling Features ─────────────────────────────────────────
lags    = [1, 2, 3, 7, 14, 30, 60, 90, 365]
windows = [7, 14, 30]                          # giữ nguyên gốc
 
for lag in lags:
    data[f'Revenue_lag_{lag}'] = data['Revenue'].shift(lag)
    data[f'COGS_lag_{lag}']    = data['COGS'].shift(lag)
 
for w in windows:
    data[f'Revenue_roll_mean_{w}'] = data['Revenue'].shift(1).rolling(w).mean()
    data[f'Revenue_roll_std_{w}']  = data['Revenue'].shift(1).rolling(w).std()
    data[f'COGS_roll_mean_{w}']    = data['COGS'].shift(1).rolling(w).mean()
    data[f'COGS_roll_std_{w}']     = data['COGS'].shift(1).rolling(w).std()
 
# BỔ SUNG: EWMA (capture momentum tốt hơn simple rolling)
for span in [7, 30]:
    data[f'Revenue_ewm_{span}'] = data['Revenue'].shift(1).ewm(span=span).mean()
    data[f'COGS_ewm_{span}']    = data['COGS'].shift(1).ewm(span=span).mean()
 
# BỔ SUNG: YoY ratio
data['Revenue_yoy'] = data['Revenue'].shift(1) / (data['Revenue'].shift(365) + 1e-6)
data['COGS_yoy']    = data['COGS'].shift(1)    / (data['COGS'].shift(365) + 1e-6)
 
data['Gross_Margin_lag_1'] = (
    (data['Revenue'].shift(1) - data['COGS'].shift(1)) /
    (data['Revenue'].shift(1) + 1)
)
 


In [7]:
# %% ── Cell 5: Expanding Mean Features (GIỮ NGUYÊN — ý tưởng rất hay) ────────
# GIỮ NGUYÊN từ notebook gốc: expanding mean không bị leakage,
# capture được "average profile" của từng tháng/tuần/dayofweek
# tích lũy từ quá khứ
 
data = data.sort_values('Date')
 
data['mean_rev_by_month'] = (
    data.groupby('month')['Revenue']
    .expanding().mean().shift(1)
    .reset_index(level=0, drop=True)
)
data['mean_cogs_by_month'] = (
    data.groupby('month')['COGS']
    .expanding().mean().shift(1)
    .reset_index(level=0, drop=True)
)
data['mean_rev_by_week'] = (
    data.groupby('weekofyear')['Revenue']
    .expanding().mean().shift(1)
    .reset_index(level=0, drop=True)
)
data['mean_cogs_by_week'] = (
    data.groupby('weekofyear')['COGS']
    .expanding().mean().shift(1)
    .reset_index(level=0, drop=True)
)
data['mean_rev_by_dayofweek'] = (
    data.groupby('dayofweek')['Revenue']
    .expanding().mean().shift(1)
    .reset_index(level=0, drop=True)
)
data['mean_cogs_by_dayofweek'] = (
    data.groupby('dayofweek')['COGS']
    .expanding().mean().shift(1)
    .reset_index(level=0, drop=True)
)
 
data = data.dropna().reset_index(drop=True)
print(f"After feature engineering: {data.shape}")
 

After feature engineering: (3468, 66)


In [8]:
# %% ── Cell 6: Feature Lists ──────────────────────────────────────────────────
calendar_features = [
    'year', 'month', 'day', 'quarter',
    'dayofweek', 'dayofyear', 'weekofyear',
    'is_weekend', 'is_holiday', 'is_tet_period',
    'is_month_start', 'is_month_end', 'is_quarter_end',
    'days_to_tet',
    'sin_1', 'cos_1', 'sin_2', 'cos_2', 'sin_dow', 'cos_dow',
]
expanding_features = [
    'mean_rev_by_month', 'mean_cogs_by_month',
    'mean_rev_by_week',  'mean_cogs_by_week',
    'mean_rev_by_dayofweek', 'mean_cogs_by_dayofweek',
]
lag_rev  = [f'Revenue_lag_{l}' for l in lags]
lag_cogs = [f'COGS_lag_{l}'    for l in lags]
roll_rev  = ([f'Revenue_roll_mean_{w}' for w in windows] +
             [f'Revenue_roll_std_{w}'  for w in windows] +
             [f'Revenue_ewm_{s}' for s in [7, 30]])
roll_cogs = ([f'COGS_roll_mean_{w}' for w in windows] +
             [f'COGS_roll_std_{w}'  for w in windows] +
             [f'COGS_ewm_{s}' for s in [7, 30]])
 
features_for_cogs = (calendar_features + expanding_features +
                     lag_cogs + roll_cogs)
features_for_revenue = (calendar_features + expanding_features +
                        lag_rev + lag_cogs + roll_rev + roll_cogs +
                        ['Gross_Margin_lag_1', 'Revenue_yoy', 'COGS_yoy'])
 
print(f"Features COGS:    {len(features_for_cogs)}")
print(f"Features Revenue: {len(features_for_revenue)}")


Features COGS:    43
Features Revenue: 63


In [9]:
# %% ── Cell 7: Train / Val Split ──────────────────────────────────────────────
train_mask = data['Date'] < '2022-01-01'
val_mask   = data['Date'] >= '2022-01-01'
 
X_train_rev  = data.loc[train_mask, features_for_revenue]
y_train_rev  = data.loc[train_mask, 'Revenue']
X_val_rev    = data.loc[val_mask,   features_for_revenue]
y_val_rev    = data.loc[val_mask,   'Revenue']
 
X_train_cogs = data.loc[train_mask, features_for_cogs]
y_train_cogs = data.loc[train_mask, 'COGS']
X_val_cogs   = data.loc[val_mask,   features_for_cogs]
y_val_cogs   = data.loc[val_mask,   'COGS']
 
print(f"Train: {train_mask.sum()} days | Validation: {val_mask.sum()} days")
 

Train: 3103 days | Validation: 365 days


In [10]:
# %% ── Cell 8: Prophet với VN Holidays ────────────────────────────────────────
# NÂNG CẤP: thêm holidays, multiplicative mode, custom seasonality
 
def fit_prophet(train_df: pd.DataFrame, target: str) -> Prophet:
    pdf = train_df[['Date', target]].rename(
        columns={'Date': 'ds', target: 'y'})
    m = Prophet(
        yearly_seasonality        = 10,
        weekly_seasonality        = 5,
        daily_seasonality         = False,
        holidays                  = vn_holidays,
        seasonality_mode          = 'multiplicative',  # tốt hơn additive với data tăng
        changepoint_prior_scale   = 0.05,
        seasonality_prior_scale   = 10.0,
        holidays_prior_scale      = 10.0,
    )
    m.add_seasonality(name='monthly',   period=30.5,  fourier_order=5)
    m.add_seasonality(name='quarterly', period=91.25, fourier_order=3)
    m.fit(pdf, seed=SEED)
    return m
 
print("Fitting Prophet Revenue (train)...")
model_prophet_rev  = fit_prophet(data[train_mask], 'Revenue')
print("Fitting Prophet COGS (train)...")
model_prophet_cogs = fit_prophet(data[train_mask], 'COGS')


Fitting Prophet Revenue (train)...


17:38:18 - cmdstanpy - INFO - Chain [1] start processing
17:38:20 - cmdstanpy - INFO - Chain [1] done processing


Fitting Prophet COGS (train)...


17:38:20 - cmdstanpy - INFO - Chain [1] start processing
17:38:21 - cmdstanpy - INFO - Chain [1] done processing


In [11]:
# %% ── Cell 9: OOF Stacking đúng chuẩn (FIX LEAKAGE) ────────────────────────
# FIX: Dùng TimeSeriesSplit để tạo OOF predictions trên TẬP TRAIN
# Meta-learner được train trên OOF → không bị leakage
# Validation set (2022) chỉ dùng để EVALUATE sau cùng, không train meta
 
def get_models():
    return {
        'lgb': LGBMRegressor(
            n_estimators=2000, learning_rate=0.02, num_leaves=64,
            subsample=0.8, colsample_bytree=0.8,
            min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
            random_state=SEED, verbose=-1, n_jobs=-1),
        'xgb': xgb.XGBRegressor(
            n_estimators=2000, learning_rate=0.02, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            min_child_weight=5, reg_alpha=0.1, reg_lambda=1.0,
            random_state=SEED, tree_method='hist', verbosity=0, n_jobs=-1),
        'cat': CatBoostRegressor(
            iterations=1000, learning_rate=0.1, depth=6,
            l2_leaf_reg=3, random_seed=SEED, verbose=0),
    }
 
def oof_train(X: pd.DataFrame, y: pd.Series,
              models_dict: dict, n_splits=5, gap=30):
    """
    TimeSeriesSplit OOF training.
    gap=30: bỏ 30 ngày giữa train và val mỗi fold để tránh lag feature leakage.
    """
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    oof  = np.zeros((len(X), len(models_dict)))
    names = list(models_dict.keys())
    print(f"  TimeSeriesSplit OOF | {n_splits} folds | gap={gap} days")
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X)):
        Xf_tr, Xf_val = X.iloc[tr_idx], X.iloc[val_idx]
        yf_tr, yf_val = y.iloc[tr_idx], y.iloc[val_idx]
        fold_mae = []
        for i, (nm, m) in enumerate(models_dict.items()):
            clf = copy.deepcopy(m)
            clf.fit(Xf_tr, yf_tr)
            p = clf.predict(Xf_val)
            oof[val_idx, i] = p
            fold_mae.append(mean_absolute_error(yf_val, p))
        print(f"  Fold {fold+1}: " +
              " | ".join(f"{n}={v:,.0f}" for n, v in zip(names, fold_mae)))
    return oof
 
def fit_final_models(X, y, models_dict, oof_preds):
    """
    1. Meta-learner (Ridge) train trên OOF predictions
    2. Base models retrain trên full (X, y)
    """
    meta = Ridge(alpha=1.0, positive=True)
    meta.fit(oof_preds, y)
    print(f"  Meta weights: { {n: round(w,4) for n, w in zip(models_dict.keys(), meta.coef_)} }")
    fitted = {}
    for nm, m in models_dict.items():
        clf = copy.deepcopy(m)
        clf.fit(X, y)
        fitted[nm] = clf
    return fitted, meta
 
def stack_predict(X, fitted_base, meta):
    bp = np.column_stack([m.predict(X) for m in fitted_base.values()])
    return meta.predict(bp)
 
# ── Train Revenue residuals ──
print("\n[Revenue] OOF trên residuals (train set)...")
prophet_rev_tr_pred  = model_prophet_rev.predict(
    pd.DataFrame({'ds': data.loc[train_mask, 'Date']}))['yhat'].values
residuals_rev_train  = y_train_rev.values - prophet_rev_tr_pred
 
models_rev = get_models()
oof_rev    = oof_train(X_train_rev,
                       pd.Series(residuals_rev_train, index=X_train_rev.index),
                       models_rev)
fitted_rev, meta_rev = fit_final_models(
    X_train_rev,
    pd.Series(residuals_rev_train, index=X_train_rev.index),
    models_rev, oof_rev)
 
# ── Train COGS residuals ──
print("\n[COGS] OOF trên residuals (train set)...")
prophet_cogs_tr_pred = model_prophet_cogs.predict(
    pd.DataFrame({'ds': data.loc[train_mask, 'Date']}))['yhat'].values
residuals_cogs_train = y_train_cogs.values - prophet_cogs_tr_pred
 
models_cogs = get_models()
oof_cogs    = oof_train(X_train_cogs,
                        pd.Series(residuals_cogs_train, index=X_train_cogs.index),
                        models_cogs)
fitted_cogs, meta_cogs = fit_final_models(
    X_train_cogs,
    pd.Series(residuals_cogs_train, index=X_train_cogs.index),
    models_cogs, oof_cogs)
 
# ── Validate trên 2022 ──
prophet_rev_val_pred  = model_prophet_rev.predict(
    pd.DataFrame({'ds': data.loc[val_mask, 'Date']}))['yhat'].values
prophet_cogs_val_pred = model_prophet_cogs.predict(
    pd.DataFrame({'ds': data.loc[val_mask, 'Date']}))['yhat'].values
 
pred_rev_val  = np.maximum(
    prophet_rev_val_pred  + stack_predict(X_val_rev,  fitted_rev,  meta_rev),  0)
pred_cogs_val = np.maximum(
    prophet_cogs_val_pred + stack_predict(X_val_cogs, fitted_cogs, meta_cogs), 0)
 
print("\n── Validation Results (2022) ──")
for label, y_true, y_pred in [
    ('Revenue (Prophet only)',    y_val_rev,  np.maximum(prophet_rev_val_pred, 0)),
    ('Revenue (Prophet + Stack)', y_val_rev,  pred_rev_val),
    ('COGS   (Prophet only)',     y_val_cogs, np.maximum(prophet_cogs_val_pred, 0)),
    ('COGS   (Prophet + Stack)',  y_val_cogs, pred_cogs_val),
]:
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"  {label:<30} MAE={mae:>12,.0f}  RMSE={rmse:>12,.0f}  R²={r2:.4f}")
 


[Revenue] OOF trên residuals (train set)...
  TimeSeriesSplit OOF | 5 folds | gap=30 days
  Fold 1: lgb=1,027,337 | xgb=997,343 | cat=997,909
  Fold 2: lgb=1,093,858 | xgb=1,053,062 | cat=1,053,806
  Fold 3: lgb=870,632 | xgb=866,773 | cat=873,765
  Fold 4: lgb=838,479 | xgb=763,441 | cat=745,054
  Fold 5: lgb=616,653 | xgb=595,923 | cat=588,967
  Meta weights: {'lgb': np.float64(0.0318), 'xgb': np.float64(0.3941), 'cat': np.float64(0.5031)}

[COGS] OOF trên residuals (train set)...
  TimeSeriesSplit OOF | 5 folds | gap=30 days
  Fold 1: lgb=883,174 | xgb=839,742 | cat=829,113
  Fold 2: lgb=890,582 | xgb=881,800 | cat=928,112
  Fold 3: lgb=743,793 | xgb=752,923 | cat=750,684
  Fold 4: lgb=645,377 | xgb=607,617 | cat=648,312
  Fold 5: lgb=518,751 | xgb=517,379 | cat=516,752
  Meta weights: {'lgb': np.float64(0.2912), 'xgb': np.float64(0.3038), 'cat': np.float64(0.2925)}

── Validation Results (2022) ──
  Revenue (Prophet only)         MAE=     986,952  RMSE=   1,350,684  R²=0.3488
  Re

In [12]:
# %% ── Cell 10: SHAP Explainability ───────────────────────────────────────────
# Bắt buộc cho báo cáo kỹ thuật (8 điểm)
print("\n[SHAP] Tính feature importance cho Revenue model (LightGBM)...")
explainer  = shap.TreeExplainer(fitted_rev['lgb'])
shap_vals  = explainer.shap_values(X_val_rev)
 
shap_df = (pd.DataFrame({
    'feature': features_for_revenue,
    'mean_abs_shap': np.abs(shap_vals).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True))
 
print("\nTop 15 features (Revenue):")
print(shap_df.head(15).to_string(index=False))
 
BUSINESS_NOTE = {
    'Revenue_lag_1'        : 'Doanh thu hôm qua — signal mạnh nhất',
    'Revenue_lag_7'        : 'Cùng ngày tuần trước — weekly pattern',
    'Revenue_lag_365'      : 'Cùng kỳ năm ngoái — YoY benchmark',
    'mean_rev_by_month'    : 'Baseline trung bình tháng (tích lũy)',
    'mean_rev_by_dayofweek': 'Profile mua sắm theo ngày trong tuần',
    'mean_rev_by_week'     : 'Baseline tuần tích lũy từ quá khứ',
    'Revenue_ewm_7'        : 'Momentum 7 ngày — phản ánh campaign',
    'Revenue_ewm_30'       : 'Momentum tháng',
    'Revenue_yoy'          : 'Tỷ lệ tăng trưởng YoY',
    'is_tet_period'        : 'Tết Nguyên Đán — ảnh hưởng lớn nhất',
    'days_to_tet'          : 'Gần Tết → spike mua sắm trước Tết',
    'is_month_end'         : 'Hành vi chi tiêu cuối tháng',
    'year'                 : 'Trend tăng trưởng theo năm',
    'Gross_Margin_lag_1'   : 'Biên lợi nhuận hôm qua',
}
 
print("\n── Business Interpretation (top 10) ──")
for _, row in shap_df.head(10).iterrows():
    note = BUSINESS_NOTE.get(row['feature'], 'Feature ảnh hưởng đến dự báo')
    print(f"  {row['feature']:<30} SHAP={row['mean_abs_shap']:>10,.0f}  →  {note}")
 
fig, ax = plt.subplots(figsize=(9, 7))
top15 = shap_df.head(15)
ax.barh(top15['feature'][::-1], top15['mean_abs_shap'][::-1],
        color='#378ADD', edgecolor='none')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Feature Importance — Revenue (SHAP)\nValidation 2022', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'shap_revenue.png'), dpi=150)
plt.close()
print("Saved: shap_revenue.png")
 


[SHAP] Tính feature importance cho Revenue model (LightGBM)...

Top 15 features (Revenue):
               feature  mean_abs_shap
                   day  166112.022975
         Revenue_lag_1  148417.418749
          COGS_lag_365  112802.692470
      COGS_roll_mean_7   80904.797605
         Revenue_lag_7   74229.898305
            COGS_lag_3   68641.330531
 mean_rev_by_dayofweek   57433.027087
            COGS_lag_7   54285.584398
     mean_rev_by_month   53386.939729
            COGS_lag_1   52640.148095
                  year   47372.888105
           Revenue_yoy   45671.278896
                 sin_1   44145.521542
mean_cogs_by_dayofweek   43713.437012
           COGS_lag_14   39497.843823

── Business Interpretation (top 10) ──
  day                            SHAP=   166,112  →  Feature ảnh hưởng đến dự báo
  Revenue_lag_1                  SHAP=   148,417  →  Doanh thu hôm qua — signal mạnh nhất
  COGS_lag_365                   SHAP=   112,803  →  Feature ảnh hưởng đến dự báo
  COGS

In [13]:
# %% ── Cell 11: Error Analysis (GIỮ NGUYÊN từ gốc + cải thiện) ───────────────
comparison = pd.DataFrame({
    'Date'            : data.loc[val_mask, 'Date'].values,
    'Revenue_actual'  : y_val_rev.values,
    'Revenue_pred'    : pred_rev_val,
    'COGS_actual'     : y_val_cogs.values,
    'COGS_pred'       : pred_cogs_val,
})
comparison['Revenue_residual'] = comparison['Revenue_actual'] - comparison['Revenue_pred']
comparison['COGS_residual']    = comparison['COGS_actual']    - comparison['COGS_pred']
comparison['abs_error_pct']    = (np.abs(comparison['Revenue_residual']) /
                                  (comparison['Revenue_actual'] + 1e-6) * 100)
comparison['month']     = comparison['Date'].dt.month
comparison['dayofweek'] = comparison['Date'].dt.dayofweek
 
print("\nMAE theo ngày trong tuần (Revenue):")
print(comparison.groupby('dayofweek')['Revenue_residual']
      .apply(lambda x: mean_absolute_error(
          comparison.loc[x.index, 'Revenue_actual'], 
          comparison.loc[x.index, 'Revenue_pred']))
      .rename({0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}))
 
print("\nMAE theo tháng (Revenue):")
print(comparison.groupby('month').apply(
    lambda g: mean_absolute_error(g['Revenue_actual'], g['Revenue_pred'])
).round(0))
 


MAE theo ngày trong tuần (Revenue):
dayofweek
Mon    672255.016835
Tue    625266.829449
Wed    770261.800719
Thu    611435.200336
Fri    692150.358743
Sat    707442.954246
Sun    660467.667624
Name: Revenue_residual, dtype: float64

MAE theo tháng (Revenue):
month
1      554211.0
2      629121.0
3     1000229.0
4      799414.0
5      778081.0
6      732307.0
7      696411.0
8      676310.0
9      760186.0
10     493452.0
11     542133.0
12     463028.0
dtype: float64


In [14]:
# %% ── Cell 12: Retrain trên FULL DATA (2012–2022) ────────────────────────────
print("\n=== Retrain trên full data (2012–2022) ===")
 
print("Refitting Prophet trên full data...")
model_prophet_rev_final  = fit_prophet(data, 'Revenue')
model_prophet_cogs_final = fit_prophet(data, 'COGS')
 
prophet_rev_all  = model_prophet_rev_final.predict(
    pd.DataFrame({'ds': data['Date']}))['yhat'].values
prophet_cogs_all = model_prophet_cogs_final.predict(
    pd.DataFrame({'ds': data['Date']}))['yhat'].values
 
residuals_rev_all  = data['Revenue'].values - prophet_rev_all
residuals_cogs_all = data['COGS'].values    - prophet_cogs_all
 
X_full_rev  = data[features_for_revenue]
X_full_cogs = data[features_for_cogs]
 
print("\n[Revenue] OOF trên full data...")
models_rev_final = get_models()
oof_rev_full = oof_train(
    X_full_rev,
    pd.Series(residuals_rev_all, index=X_full_rev.index),
    models_rev_final)
fitted_rev_final, meta_rev_final = fit_final_models(
    X_full_rev,
    pd.Series(residuals_rev_all, index=X_full_rev.index),
    models_rev_final, oof_rev_full)
 
print("\n[COGS] OOF trên full data...")
models_cogs_final = get_models()
oof_cogs_full = oof_train(
    X_full_cogs,
    pd.Series(residuals_cogs_all, index=X_full_cogs.index),
    models_cogs_final)
fitted_cogs_final, meta_cogs_final = fit_final_models(
    X_full_cogs,
    pd.Series(residuals_cogs_all, index=X_full_cogs.index),
    models_cogs_final, oof_cogs_full)
 
print("Full retrain complete.")
 


=== Retrain trên full data (2012–2022) ===
Refitting Prophet trên full data...


17:47:01 - cmdstanpy - INFO - Chain [1] start processing
17:47:02 - cmdstanpy - INFO - Chain [1] done processing
17:47:02 - cmdstanpy - INFO - Chain [1] start processing
17:47:03 - cmdstanpy - INFO - Chain [1] done processing



[Revenue] OOF trên full data...
  TimeSeriesSplit OOF | 5 folds | gap=30 days
  Fold 1: lgb=1,117,245 | xgb=1,047,522 | cat=1,049,252
  Fold 2: lgb=905,746 | xgb=830,710 | cat=856,944
  Fold 3: lgb=905,758 | xgb=872,544 | cat=876,823
  Fold 4: lgb=749,455 | xgb=706,556 | cat=701,341
  Fold 5: lgb=633,778 | xgb=632,898 | cat=622,277
  Meta weights: {'lgb': np.float64(0.0), 'xgb': np.float64(0.6005), 'cat': np.float64(0.3621)}

[COGS] OOF trên full data...
  TimeSeriesSplit OOF | 5 folds | gap=30 days
  Fold 1: lgb=926,854 | xgb=878,217 | cat=876,788
  Fold 2: lgb=754,392 | xgb=751,781 | cat=793,071
  Fold 3: lgb=782,654 | xgb=752,431 | cat=722,379
  Fold 4: lgb=618,637 | xgb=591,522 | cat=606,664
  Fold 5: lgb=567,224 | xgb=565,065 | cat=559,089
  Meta weights: {'lgb': np.float64(0.0), 'xgb': np.float64(0.5007), 'cat': np.float64(0.4143)}
Full retrain complete.


In [15]:
# %% ── Cell 13: Recursive Forecast ───────────────────────────────────────────
# FIX: Tính đúng expanding mean features cho mỗi ngày forecast
# GIỮ: Cấu trúc recursive từ notebook gốc
 
def recursive_forecast(historical_data, forecast_dates,
                       prophet_rev_m, prophet_cogs_m,
                       fitted_rev_m, meta_rev_m,
                       fitted_cogs_m, meta_cogs_m,
                       features_for_rev, features_for_cogs,
                       lags, windows,
                       # expanding mean state từ cuối train
                       expanding_state: dict):
    """
    FIX QUAN TRỌNG: Tính đúng expanding mean features trong recursive loop.
    Notebook gốc bỏ sót features này → model nhận 0 cho toàn bộ test dates.
    """
    hist = historical_data.copy().reset_index(drop=True)
    predictions = []
 
    for forecast_date in forecast_dates:
        f = {}
        rev_s  = hist['Revenue']
        cogs_s = hist['COGS']
 
        # ── Calendar ──
        f['year']          = forecast_date.year
        f['month']         = forecast_date.month
        f['day']           = forecast_date.day
        f['quarter']       = (forecast_date.month - 1) // 3 + 1
        f['dayofweek']     = forecast_date.dayofweek
        f['dayofyear']     = forecast_date.timetuple().tm_yday
        f['weekofyear']    = int(forecast_date.isocalendar().week)
        f['is_weekend']    = int(forecast_date.dayofweek in [5, 6])
        f['is_holiday']    = int(forecast_date.strftime('%m-%d') in HOLIDAYS_MMDD)
        f['is_tet_period'] = int(forecast_date in TET_WINDOW)
        f['is_month_start']= int(forecast_date.day == 1)
        f['is_month_end']  = int((forecast_date + pd.Timedelta(1)).month != forecast_date.month)
        f['is_quarter_end']= int(f['is_month_end'] and forecast_date.month in [3,6,9,12])
        f['days_to_tet']   = int((tet_ts - forecast_date).abs().min().days)
 
        doy = f['dayofyear']; dow = f['dayofweek']
        f['sin_1']   = np.sin(2*np.pi*doy/365.25)
        f['cos_1']   = np.cos(2*np.pi*doy/365.25)
        f['sin_2']   = np.sin(4*np.pi*doy/365.25)
        f['cos_2']   = np.cos(4*np.pi*doy/365.25)
        f['sin_dow'] = np.sin(2*np.pi*dow/7)
        f['cos_dow'] = np.cos(2*np.pi*dow/7)
 
        # ── FIX: Expanding mean features ──────────────────────────────
        # Update state với ngày mới nhất của historical (chưa include hôm nay)
        last_rev  = float(rev_s.iloc[-1])
        last_cogs = float(cogs_s.iloc[-1])
        mon = f['month']; woy = f['weekofyear']; dow_ = f['dayofweek']
 
        # Update expanding counts
        expanding_state['count_by_month'][mon]    = expanding_state['count_by_month'].get(mon, 0) + 1
        expanding_state['sum_rev_by_month'][mon]  = expanding_state['sum_rev_by_month'].get(mon, 0.0) + last_rev
        expanding_state['sum_cogs_by_month'][mon] = expanding_state['sum_cogs_by_month'].get(mon, 0.0) + last_cogs
        expanding_state['count_by_week'][woy]     = expanding_state['count_by_week'].get(woy, 0) + 1
        expanding_state['sum_rev_by_week'][woy]   = expanding_state['sum_rev_by_week'].get(woy, 0.0) + last_rev
        expanding_state['sum_cogs_by_week'][woy]  = expanding_state['sum_cogs_by_week'].get(woy, 0.0) + last_cogs
        expanding_state['count_by_dow'][dow_]     = expanding_state['count_by_dow'].get(dow_, 0) + 1
        expanding_state['sum_rev_by_dow'][dow_]   = expanding_state['sum_rev_by_dow'].get(dow_, 0.0) + last_rev
        expanding_state['sum_cogs_by_dow'][dow_]  = expanding_state['sum_cogs_by_dow'].get(dow_, 0.0) + last_cogs
 
        f['mean_rev_by_month']  = (expanding_state['sum_rev_by_month'][mon] /
                                   expanding_state['count_by_month'][mon])
        f['mean_cogs_by_month'] = (expanding_state['sum_cogs_by_month'][mon] /
                                   expanding_state['count_by_month'][mon])
        f['mean_rev_by_week']   = (expanding_state['sum_rev_by_week'].get(woy, rev_s.mean()) /
                                   max(expanding_state['count_by_week'].get(woy, 1), 1))
        f['mean_cogs_by_week']  = (expanding_state['sum_cogs_by_week'].get(woy, cogs_s.mean()) /
                                   max(expanding_state['count_by_week'].get(woy, 1), 1))
        f['mean_rev_by_dayofweek']  = (expanding_state['sum_rev_by_dow'][dow_] /
                                       expanding_state['count_by_dow'][dow_])
        f['mean_cogs_by_dayofweek'] = (expanding_state['sum_cogs_by_dow'][dow_] /
                                       expanding_state['count_by_dow'][dow_])
 
        # ── Lag & Rolling ──
        for lag in lags:
            idx = len(rev_s) - lag
            f[f'Revenue_lag_{lag}'] = float(rev_s.iloc[idx])  if idx >= 0 else float(rev_s.mean())
            f[f'COGS_lag_{lag}']    = float(cogs_s.iloc[idx]) if idx >= 0 else float(cogs_s.mean())
 
        for w in windows:
            tr = rev_s.iloc[-w:]  if len(rev_s)  >= w else rev_s
            tc = cogs_s.iloc[-w:] if len(cogs_s) >= w else cogs_s
            f[f'Revenue_roll_mean_{w}'] = float(tr.mean())
            f[f'Revenue_roll_std_{w}']  = float(tr.std()) if len(tr) > 1 else 0.0
            f[f'COGS_roll_mean_{w}']    = float(tc.mean())
            f[f'COGS_roll_std_{w}']     = float(tc.std()) if len(tc) > 1 else 0.0
 
        for span in [7, 30]:
            f[f'Revenue_ewm_{span}'] = float(rev_s.ewm(span=span).mean().iloc[-1])
            f[f'COGS_ewm_{span}']    = float(cogs_s.ewm(span=span).mean().iloc[-1])
 
        r1 = float(rev_s.iloc[-1]); c1 = float(cogs_s.iloc[-1])
        f['Gross_Margin_lag_1'] = (r1 - c1) / (r1 + 1)
        r365 = float(rev_s.iloc[-365])  if len(rev_s) >= 365  else float(rev_s.mean())
        c365 = float(cogs_s.iloc[-365]) if len(cogs_s) >= 365 else float(cogs_s.mean())
        f['Revenue_yoy'] = r1 / (r365 + 1e-6)
        f['COGS_yoy']    = c1 / (c365 + 1e-6)
 
        # ── Prophet predict ──
        future_df = pd.DataFrame({'ds': [forecast_date]})
        p_cogs = float(prophet_cogs_m.predict(future_df)['yhat'].values[0])
        p_rev  = float(prophet_rev_m.predict(future_df)['yhat'].values[0])
 
        # ── Predict COGS ──
        X_c = pd.DataFrame([f]).reindex(columns=features_for_cogs, fill_value=0.0)
        bp_c = np.column_stack([m.predict(X_c) for m in fitted_cogs_m.values()])
        pred_cogs = max(0.0, p_cogs + float(meta_cogs_m.predict(bp_c)[0]))
 
        # ── Predict Revenue ──
        X_r = pd.DataFrame([f]).reindex(columns=features_for_rev, fill_value=0.0)
        bp_r = np.column_stack([m.predict(X_r) for m in fitted_rev_m.values()])
        pred_rev = max(0.0, p_rev + float(meta_rev_m.predict(bp_r)[0]))
 
        predictions.append({'Date': forecast_date,
                             'Revenue': pred_rev, 'COGS': pred_cogs})
 
        # Update history
        new_row = pd.DataFrame([{'Date': forecast_date,
                                  'Revenue': pred_rev, 'COGS': pred_cogs}])
        hist = pd.concat([hist, new_row], ignore_index=True)
 
    return pd.DataFrame(predictions)
 
 
def build_expanding_state(df_hist: pd.DataFrame) -> dict:
    """Build expanding mean state từ historical data để tiếp tục trong recursive."""
    state = {
        'count_by_month': {}, 'sum_rev_by_month': {}, 'sum_cogs_by_month': {},
        'count_by_week':  {}, 'sum_rev_by_week':  {}, 'sum_cogs_by_week':  {},
        'count_by_dow':   {}, 'sum_rev_by_dow':   {}, 'sum_cogs_by_dow':   {},
    }
    for _, row in df_hist.iterrows():
        mon = row['Date'].month
        woy = int(row['Date'].isocalendar().week)
        dow = row['Date'].dayofweek
        r   = row['Revenue']; c = row['COGS']
        state['count_by_month'][mon]    = state['count_by_month'].get(mon, 0) + 1
        state['sum_rev_by_month'][mon]  = state['sum_rev_by_month'].get(mon, 0.0) + r
        state['sum_cogs_by_month'][mon] = state['sum_cogs_by_month'].get(mon, 0.0) + c
        state['count_by_week'][woy]     = state['count_by_week'].get(woy, 0) + 1
        state['sum_rev_by_week'][woy]   = state['sum_rev_by_week'].get(woy, 0.0) + r
        state['sum_cogs_by_week'][woy]  = state['sum_cogs_by_week'].get(woy, 0.0) + c
        state['count_by_dow'][dow]      = state['count_by_dow'].get(dow, 0) + 1
        state['sum_rev_by_dow'][dow]    = state['sum_rev_by_dow'].get(dow, 0.0) + r
        state['sum_cogs_by_dow'][dow]   = state['sum_cogs_by_dow'].get(dow, 0.0) + c
    return state
 

In [16]:
# %% ── Cell 14: Validate Recursive trên 2022 ─────────────────────────────────
print("\n=== Validate Recursive trên 2022 ===")
train_data  = data[train_mask][['Date','Revenue','COGS']].reset_index(drop=True)
exp_state_val = build_expanding_state(train_data)
 
recursive_val = recursive_forecast(
    train_data,
    pd.date_range('2022-01-01', '2022-12-31', freq='D'),
    model_prophet_rev, model_prophet_cogs,
    fitted_rev, meta_rev,
    fitted_cogs, meta_cogs,
    features_for_revenue, features_for_cogs,
    lags, windows, exp_state_val
)
 
comp = recursive_val.merge(
    data.loc[val_mask, ['Date','Revenue','COGS']],
    on='Date', suffixes=('_pred','_actual')
)
print(f"Revenue  RMSE={np.sqrt(mean_squared_error(comp.Revenue_actual, comp.Revenue_pred)):,.0f}"
      f"  MAE={mean_absolute_error(comp.Revenue_actual, comp.Revenue_pred):,.0f}"
      f"  R²={r2_score(comp.Revenue_actual, comp.Revenue_pred):.4f}")
print(f"COGS     RMSE={np.sqrt(mean_squared_error(comp.COGS_actual, comp.COGS_pred)):,.0f}"
      f"  MAE={mean_absolute_error(comp.COGS_actual, comp.COGS_pred):,.0f}"
      f"  R²={r2_score(comp.COGS_actual, comp.COGS_pred):.4f}")
 


=== Validate Recursive trên 2022 ===
Revenue  RMSE=977,781  MAE=735,169  R²=0.6588
COGS     RMSE=862,081  MAE=659,153  R²=0.6507


In [17]:
# %% ── Cell 15: Generate Test Submission ──────────────────────────────────────
print("\n=== Generating Test Predictions (2023-01-01 → 2024-07-01) ===")
full_hist     = data[['Date','Revenue','COGS']].reset_index(drop=True)
exp_state_test = build_expanding_state(full_hist)
 
test_dates = pd.date_range('2023-01-01', '2024-07-01', freq='D')
submission = recursive_forecast(
    full_hist, test_dates,
    model_prophet_rev_final, model_prophet_cogs_final,
    fitted_rev_final, meta_rev_final,
    fitted_cogs_final, meta_cogs_final,
    features_for_revenue, features_for_cogs,
    lags, windows, exp_state_test
)
 
# Align với sample_submission
submission['Date'] = submission['Date'].dt.strftime('%Y-%m-%d')
sample_sub['Date'] = sample_sub['Date'].dt.strftime('%Y-%m-%d')
final_sub = sample_sub[['Date']].merge(submission, on='Date', how='left')
final_sub = final_sub[['Date','Revenue','COGS']]
 
assert len(final_sub) == len(sample_sub), "Row count mismatch!"
out = os.path.join(OUT_DIR, 'submission_v2.csv')
final_sub.to_csv(out, index=False)
print(f"Saved: {out}")
print(f"Shape: {final_sub.shape}")
print(final_sub.head(10).to_string(index=False))


=== Generating Test Predictions (2023-01-01 → 2024-07-01) ===
Saved: ../data/outputs\submission_v2.csv
Shape: (548, 3)
      Date      Revenue         COGS
2023-01-01 2.003832e+06 2.049982e+06
2023-01-02 1.535014e+06 1.310754e+06
2023-01-03 1.804710e+06 1.707804e+06
2023-01-04 2.463556e+06 1.952500e+06
2023-01-05 2.056129e+06 1.845896e+06
2023-01-06 1.627687e+06 1.532813e+06
2023-01-07 1.273485e+06 1.050565e+06
2023-01-08 1.141912e+06 1.091188e+06
2023-01-09 9.826227e+05 1.036120e+06
2023-01-10 1.228400e+06 1.091074e+06


In [18]:
# %% ── Cell 16: Visualizations (GIỮ NGUYÊN từ gốc + cải thiện) ───────────────
fig, axes = plt.subplots(3, 1, figsize=(15, 14), facecolor='white')
fig.suptitle('DATATHON 2026 — Revenue Forecasting v2', fontsize=14, fontweight='bold')
 
# Actual vs Predicted (Validation)
axes[0].plot(data['Date'], data['Revenue'],
             color='#4477AA', linewidth=0.7, alpha=0.7, label='Actual Train')
axes[0].plot(comp['Date'], comp['Revenue_pred'],
             color='#EE6677', linewidth=1.5, label='Predicted (recursive, 2022)')
axes[0].set_title('Actual vs Predicted Revenue — Validation 2022', fontweight='bold')
axes[0].set_ylabel('Revenue'); axes[0].legend()
 
# Full forecast
axes[1].plot(data['Date'], data['Revenue'],
             color='#4477AA', linewidth=0.7, alpha=0.7, label='Actual')
axes[1].plot(pd.to_datetime(final_sub['Date']), final_sub['Revenue'],
             color='#EE6677', linewidth=1.5, label='Forecast 2023–2024')
axes[1].axvline(pd.Timestamp('2023-01-01'), color='gray',
                linestyle='--', linewidth=1, alpha=0.7)
axes[1].set_title('Revenue Forecast — Test Period', fontweight='bold')
axes[1].set_ylabel('Revenue'); axes[1].legend()
 
# Residuals
comp['rev_residual'] = comp['Revenue_actual'] - comp['Revenue_pred']
axes[2].scatter(comp['Date'], comp['rev_residual'], s=8, alpha=0.5, color='#AA3377')
axes[2].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[2].set_title('Residuals — Validation 2022', fontweight='bold')
axes[2].set_ylabel('Actual − Predicted')
 
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'forecast_results_v2.png'), dpi=150, bbox_inches='tight')
plt.close()
 
print("\n" + "="*60)
print("DONE! Output files:")
print(f"  {OUT_DIR}/submission_v2.csv")
print(f"  {OUT_DIR}/shap_revenue.png")
print(f"  {OUT_DIR}/forecast_results_v2.png")
print("="*60)
 


DONE! Output files:
  ../data/outputs/submission_v2.csv
  ../data/outputs/shap_revenue.png
  ../data/outputs/forecast_results_v2.png
